# Instagram Usage & Psychological Risk — Random Forest Models

Trains and evaluates two Random Forest models on the combined dataset:
- **Regressor**: predict continuous `perceived_stress_score`
- **Classifier**: predict binary `high_stress` (top 20% of scores)

All plots → `../results/random_forest/`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from pathlib import Path

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    roc_auc_score, accuracy_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, CalibrationDisplay,
)
from sklearn.inspection import PartialDependenceDisplay

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#F8F9FA',
    'axes.edgecolor': '#CCCCCC',
    'axes.grid': True,
    'grid.color': 'white',
    'grid.linewidth': 0.9,
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.titlesize': 15,
    'figure.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': '#CCCCCC',
    'axes.spines.bottom': '#CCCCCC',
})

PALETTE  = sns.color_palette('Set2')
PALETTE2 = sns.color_palette('tab10')

TARGET      = 'perceived_stress_score'
TARGET_BIN  = 'high_stress'
STRESS_PCT  = 0.80   # top 20% = high stress

RESULTS = Path('../results/random_forest')
RESULTS.mkdir(parents=True, exist_ok=True)
print(f'Saving plots to: {RESULTS.resolve()}')

## 1. Load & Combine Both Datasets

In [ ]:
print('Loading dataset 1 ...')
df1 = pd.read_csv('../dataset/instagram_usage_lifestyle.csv', low_memory=False)
print(f'  {df1.shape}')

print('Loading dataset 2 ...')
df2 = pd.read_csv('../dataset/instagram_users_lifestyle.csv', low_memory=False)
print(f'  {df2.shape}')

df_full = pd.concat([df1, df2], ignore_index=True)
if 'user_id' in df_full.columns:
    df_full = df_full.drop_duplicates(subset='user_id')
else:
    df_full = df_full.drop_duplicates()

print(f'Combined: {df_full.shape}')

# Sample for model training — large enough for stable estimates
SAMPLE = min(400_000, len(df_full))
df = df_full.sample(n=SAMPLE, random_state=42).reset_index(drop=True)
print(f'Training sample: {df.shape}')

## 2. Prepare Features

In [ ]:
# Drop non-informative / leakage columns
drop_cols = ['user_id', 'app_name', 'country', 'last_login_date',
             'content_type_preference', 'preferred_content_theme',
             'self_reported_happiness']

df_m = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore').copy()

# Encode string/bool columns
le = LabelEncoder()
for col in df_m.select_dtypes(include=['object', 'category']).columns:
    if col != TARGET:
        df_m[col] = le.fit_transform(df_m[col].fillna('Unknown').astype(str))
for col in df_m.select_dtypes(include='bool').columns:
    df_m[col] = df_m[col].astype(int)

df_m = df_m.fillna(df_m.median(numeric_only=True))

# Binary target
threshold = df_m[TARGET].quantile(STRESS_PCT)
df_m[TARGET_BIN] = (df_m[TARGET] >= threshold).astype(int)
print(f'High-stress threshold (80th pct): {threshold:.2f}')
print(f'High-stress prevalence: {df_m[TARGET_BIN].mean():.1%}')

feature_cols = [c for c in df_m.columns if c not in [TARGET, TARGET_BIN]]
print(f'\nFeatures: {len(feature_cols)}')
print(feature_cols)

## 3. Train / Test Split

In [ ]:
X = df_m[feature_cols].values
y_reg = df_m[TARGET].values
y_clf = df_m[TARGET_BIN].values

X_train, X_test, yr_train, yr_test, yc_train, yc_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

print(f'Training set : {X_train.shape[0]:,} rows')
print(f'Test set     : {X_test.shape[0]:,} rows')

## 4. Random Forest Regressor

In [ ]:
print('Fitting RandomForestRegressor ...')
rf_reg = RandomForestRegressor(
    n_estimators=200,
    max_depth=16,
    min_samples_leaf=10,
    max_features='sqrt',
    oob_score=True,
    random_state=42,
    n_jobs=-1,
)
rf_reg.fit(X_train, yr_train)

yr_pred = rf_reg.predict(X_test)

rmse = np.sqrt(mean_squared_error(yr_test, yr_pred))
mae  = mean_absolute_error(yr_test, yr_pred)
r2   = r2_score(yr_test, yr_pred)
oob  = rf_reg.oob_score_

print(f'\n=== Regression Metrics ===')
print(f'  RMSE    : {rmse:.4f}')
print(f'  MAE     : {mae:.4f}')
print(f'  R²      : {r2:.4f}')
print(f'  OOB R²  : {oob:.4f}')

# Save metrics
reg_metrics = pd.DataFrame({'Metric': ['RMSE', 'MAE', 'R2', 'OOB_R2'],
                             'Value':  [rmse, mae, r2, oob]})
reg_metrics.to_csv(RESULTS / 'rf_regression_metrics.csv', index=False)

In [ ]:
# Residual analysis
residuals = yr_test - yr_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Random Forest Regressor — Residual Analysis')

# Predicted vs Actual
sample_n = min(8000, len(yr_test))
idx_s = np.random.choice(len(yr_test), sample_n, replace=False)
axes[0].scatter(yr_test[idx_s], yr_pred[idx_s], alpha=0.15, s=8, color=PALETTE2[0])
mn, mx = yr_test.min(), yr_test.max()
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect fit')
axes[0].set_xlabel('Actual Stress Score')
axes[0].set_ylabel('Predicted Stress Score')
axes[0].set_title(f'Predicted vs Actual  (R² = {r2:.3f})')
axes[0].legend()

# Residuals vs Predicted
axes[1].scatter(yr_pred[idx_s], residuals[idx_s], alpha=0.15, s=8, color=PALETTE2[1])
axes[1].axhline(0, color='#C0392B', lw=1.5, linestyle='--')
axes[1].set_xlabel('Predicted Stress Score')
axes[1].set_ylabel('Residual (Actual − Predicted)')
axes[1].set_title('Residuals vs Predicted')

# Residual distribution
axes[2].hist(residuals, bins=60, color=PALETTE2[2], edgecolor='white', alpha=0.85,
             density=True)
pd.Series(residuals).plot.kde(ax=axes[2], color='#C0392B', lw=2, label='KDE')
axes[2].set_xlabel('Residual')
axes[2].set_ylabel('Density')
axes[2].set_title('Residual Distribution')
axes[2].legend()

fig.tight_layout()
fig.savefig(RESULTS / 'rf_regression_residuals.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: rf_regression_residuals.png')

## 5. Random Forest Classifier (High Stress)

In [ ]:
print('Fitting RandomForestClassifier ...')
rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=16,
    min_samples_leaf=10,
    max_features='sqrt',
    oob_score=True,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
rf_clf.fit(X_train, yc_train)

yc_pred      = rf_clf.predict(X_test)
yc_prob      = rf_clf.predict_proba(X_test)[:, 1]

auc  = roc_auc_score(yc_test, yc_prob)
acc  = accuracy_score(yc_test, yc_pred)
f1   = f1_score(yc_test, yc_pred)
oob_clf = rf_clf.oob_score_

print(f'\n=== Classification Metrics ===')
print(f'  AUC-ROC  : {auc:.4f}')
print(f'  Accuracy : {acc:.4f}')
print(f'  F1 Score : {f1:.4f}')
print(f'  OOB Acc  : {oob_clf:.4f}')

clf_metrics = pd.DataFrame({'Metric': ['AUC', 'Accuracy', 'F1', 'OOB_Accuracy'],
                             'Value':  [auc, acc, f1, oob_clf]})
clf_metrics.to_csv(RESULTS / 'rf_classification_metrics.csv', index=False)

In [ ]:
# Confusion matrix + ROC + Calibration
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Random Forest Classifier — Evaluation')

# Confusion matrix
cm = confusion_matrix(yc_test, yc_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Low Stress', 'High Stress'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix\n(Accuracy = {acc:.3f})')

# ROC curve
RocCurveDisplay.from_predictions(yc_test, yc_prob, ax=axes[1],
                                  color=PALETTE2[0], lw=2)
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[1].set_title(f'ROC Curve  (AUC = {auc:.3f})')
axes[1].legend()
axes[1].set_facecolor('#F8F9FA')

# Calibration curve
CalibrationDisplay.from_predictions(yc_test, yc_prob,
                                     n_bins=10, ax=axes[2],
                                     color=PALETTE2[1], lw=2)
axes[2].set_title('Calibration Curve')
axes[2].set_facecolor('#F8F9FA')

fig.tight_layout()
fig.savefig(RESULTS / 'rf_classification_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: rf_classification_evaluation.png')

## 6. Feature Importance

In [ ]:
def plot_importance(importances, feature_names, title, fname, top_n=20):
    imp_series = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(top_n)

    fig, ax = plt.subplots(figsize=(11, max(6, top_n * 0.42)))
    y_pos = np.arange(len(imp_series))
    colors = PALETTE2[:top_n] * (top_n // len(PALETTE2) + 1)

    bars = ax.barh(y_pos, imp_series.values, color=colors[:top_n],
                   edgecolor='white', height=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(imp_series.index, fontsize=9)
    ax.set_xlabel('Feature Importance (mean impurity decrease)')
    ax.set_title(title)
    ax.invert_yaxis()

    # Percentage labels
    total = imp_series.sum()
    for bar, val in zip(bars, imp_series.values):
        ax.text(bar.get_width() + total * 0.002,
                bar.get_y() + bar.get_height() / 2,
                f'{val / total:.1%}', va='center', fontsize=8)

    ax.set_xlim(0, imp_series.max() * 1.18)

    fig.tight_layout()
    fig.savefig(RESULTS / fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname}')
    return imp_series


reg_imp = plot_importance(
    rf_reg.feature_importances_, feature_cols,
    title='Random Forest Regressor — Feature Importance\n(Top 20, % of total importance)',
    fname='rf_regression_feature_importance.png',
)

clf_imp = plot_importance(
    rf_clf.feature_importances_, feature_cols,
    title='Random Forest Classifier — Feature Importance\n(Top 20, % of total importance)',
    fname='rf_classification_feature_importance.png',
)

## 7. Partial Dependence Plots — Regressor

In [ ]:
# Pick the 6 most important features for PDP
top_pdp_feats = reg_imp.head(6).index.tolist()
feat_indices  = [feature_cols.index(f) for f in top_pdp_feats]

# Use a subsample for PDP speed
pdp_n = min(20_000, len(X_train))
rng   = np.random.default_rng(99)
pdp_idx = rng.choice(len(X_train), pdp_n, replace=False)
X_pdp = X_train[pdp_idx]

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle('Partial Dependence Plots — RF Regressor\n(marginal effect of each feature on predicted stress)')

PartialDependenceDisplay.from_estimator(
    rf_reg, X_pdp, feat_indices,
    feature_names=feature_cols,
    ax=axes.ravel()[:len(feat_indices)],
    line_kw={'color': PALETTE2[0], 'lw': 2.2},
    pd_line_kw={'color': PALETTE2[0]},
    ice_lines_kw={'alpha': 0.05, 'color': '#888888'},
    kind='average',
)

for ax in axes.ravel():
    ax.set_facecolor('#F8F9FA')

fig.tight_layout()
fig.savefig(RESULTS / 'rf_partial_dependence_regression.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: rf_partial_dependence_regression.png')

## 8. Partial Dependence Plots — Classifier

In [ ]:
top_clf_feats = clf_imp.head(6).index.tolist()
clf_feat_idx  = [feature_cols.index(f) for f in top_clf_feats]

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle('Partial Dependence Plots — RF Classifier\n(marginal effect on P(high stress))')

PartialDependenceDisplay.from_estimator(
    rf_clf, X_pdp, clf_feat_idx,
    feature_names=feature_cols,
    ax=axes.ravel()[:len(clf_feat_idx)],
    line_kw={'color': PALETTE2[3], 'lw': 2.2},
    kind='average',
    response_method='predict_proba',
)

for ax in axes.ravel():
    ax.set_facecolor('#F8F9FA')

fig.tight_layout()
fig.savefig(RESULTS / 'rf_partial_dependence_classification.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: rf_partial_dependence_classification.png')

## 9. Feature Importance Comparison (Regressor vs Classifier)

In [ ]:
# Align top-20 by regression importance
top20_reg = reg_imp.head(20).index.tolist()
reg_vals  = pd.Series(rf_reg.feature_importances_, index=feature_cols).loc[top20_reg]
clf_vals  = pd.Series(rf_clf.feature_importances_, index=feature_cols).loc[top20_reg]

# Normalise within each model
reg_norm = reg_vals / reg_vals.sum()
clf_norm = clf_vals / clf_vals.sum()

x_pos = np.arange(len(top20_reg))
width = 0.4

fig, ax = plt.subplots(figsize=(14, 7))
ax.bar(x_pos - width / 2, reg_norm.values, width=width,
       label='Regressor', color=PALETTE2[0], edgecolor='white', alpha=0.88)
ax.bar(x_pos + width / 2, clf_norm.values, width=width,
       label='Classifier', color=PALETTE2[3], edgecolor='white', alpha=0.88)

ax.set_xticks(x_pos)
ax.set_xticklabels([f.replace('_', '\n') for f in top20_reg], fontsize=8, rotation=0)
ax.set_ylabel('Normalised Feature Importance')
ax.set_title('Feature Importance Comparison — Regressor vs Classifier\n'
             '(normalised within each model, top 20 by regression importance)')
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

fig.tight_layout()
fig.savefig(RESULTS / 'rf_importance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: rf_importance_comparison.png')

## 10. Model Performance Summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Random Forest — Model Performance Summary')

# Regression metrics bar
reg_labels  = ['RMSE', 'MAE', 'R²', 'OOB R²']
reg_values  = [rmse, mae, r2, oob]
colours_reg = [PALETTE[1], PALETTE[2], PALETTE[0], PALETTE[3]]

bars_r = axes[0].bar(reg_labels, reg_values, color=colours_reg, edgecolor='white', width=0.6)
for bar, val in zip(bars_r, reg_values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('Regression  (target: stress score)')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, max(reg_values) * 1.2)

# Classification metrics bar
clf_labels  = ['AUC-ROC', 'Accuracy', 'F1 Score', 'OOB Acc']
clf_values  = [auc, acc, f1, oob_clf]
colours_clf = [PALETTE[0], PALETTE[1], PALETTE[2], PALETTE[3]]

bars_c = axes[1].bar(clf_labels, clf_values, color=colours_clf, edgecolor='white', width=0.6)
for bar, val in zip(bars_c, clf_values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_title('Classification  (target: high stress, top 20%)')
axes[1].set_ylabel('Score')
axes[1].set_ylim(0, 1.15)

fig.tight_layout()
fig.savefig(RESULTS / 'rf_performance_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== FINAL MODEL SUMMARY ===')
print(f'Regression  — RMSE: {rmse:.3f}  MAE: {mae:.3f}  R²: {r2:.3f}  OOB: {oob:.3f}')
print(f'Classifier  — AUC:  {auc:.3f}  Acc: {acc:.3f}  F1: {f1:.3f}  OOB: {oob_clf:.3f}')
print('\nAll plots saved to:', RESULTS.resolve())